# Chapter 57: PatchTST for Time Series and HAR

How do patch length and overlapping stride change tokens, coverage, and attention-pair count?

The chapter teaches me to turn local signal windows into manageable attention tokens. The unfolding code is concrete enough to follow with Python-array reasoning. I would reread the token reduction claim because overlapping stride changes the count. I also need to know whether the forecasting head emits one target or every channel. The companion isolates patch construction and coverage. Patch a 48-point constructed signal with stride half the selected even patch length. Compute patch starts and means, count how often each time point is covered, and compare squared token counts against 48 single-step tokens. Report uncovered tail points and overlap multiplicity. Overlapping patches generally produce more than L/P tokens. Coverage can leave a tail when division is uneven. Token-pair reduction describes one attention term; the projection and prediction-head costs remain separate. Compare patch lengths four and sixteen. Calculate tokens using the displayed stride rather than length divided by patch size, and explain how uncovered tail samples would affect real signal preparation.

This is a constructed teaching example. Its results describe these supplied inputs, not a measured competition gain.

## Assumptions

Even patch lengths 4 to 16 and stride=P/2. Patch means stand in for inspection, not learned token projections. Attention-pair count is a per-channel illustration, not whole-model runtime.

In [1]:
parameter = 8
# Change this input, then run the following calculation.

In [2]:
import math, json
assert int(parameter)==parameter and 4<=parameter<=16 and int(parameter)%2==0
p=int(parameter); stride=p//2; signal=[math.sin(2*math.pi*t/12)+.03*t for t in range(48)]
starts=list(range(0,49-p,stride)); cover=[0]*48; rows=[]
for start in starts:
 for t in range(start,start+p): cover[t]+=1
 rows.append({'start':start,'end_exclusive':start+p,'mean':sum(signal[start:start+p])/p})
assert len(starts)==(48-p)//stride+1
print(json.dumps({'patch_length':p,'stride':stride,'tokens':len(starts),'single_step_pairs':48**2,'patch_pairs':len(starts)**2,'uncovered_points':sum(c==0 for c in cover),'maximum_coverage':max(cover),'table':rows}))


{"patch_length": 8, "stride": 4, "tokens": 11, "single_step_pairs": 2304, "patch_pairs": 121, "uncovered_points": 0, "maximum_coverage": 2, "table": [{"start": 0, "end_exclusive": 8, "mean": 0.5090063509461097}, {"start": 4, "end_exclusive": 12, "mean": -0.07075317547305482}, {"start": 8, "end_exclusive": 16, "mean": 0.23674682452694507}, {"start": 12, "end_exclusive": 20, "mean": 0.8690063509461098}, {"start": 16, "end_exclusive": 24, "mean": 0.2892468245269451}, {"start": 20, "end_exclusive": 28, "mean": 0.5967468245269449}, {"start": 24, "end_exclusive": 32, "mean": 1.2290063509461102}, {"start": 28, "end_exclusive": 36, "mean": 0.6492468245269458}, {"start": 32, "end_exclusive": 40, "mean": 0.9567468245269449}, {"start": 36, "end_exclusive": 44, "mean": 1.5890063509461099}, {"start": 40, "end_exclusive": 48, "mean": 1.0092468245269457}]}


## Interpretation

Overlapping patches generally produce more than L/P tokens. Coverage can leave a tail when division is uneven. Token-pair reduction describes one attention term; the projection and prediction-head costs remain separate.

Book location: Chapter 57, ## The Architecture: Patches, Position, and Representation.

[Technical corrections](https://github.com/KarpelesPublishing/winning-kaggle-companion/blob/main/docs/errata.md).